TO-DO
- Interface für Normies
- Protokolldatei als Text für Veränderungen. Datum, Importiert, etc.


In [ ]:
%pip install ifcopenshell

# Datenbank aus IFC anlegen oder updaten

In [ ]:
# Processes an IFC file with ProvisionForVoids to create or update a sqlite database
import ifcopenshell
import sqlite3
import os
from datetime import datetime

# Define the path to the updated IFC file
# Replace with the actual path to your updated IFC file
updated_file_path = "SuD_Aussparungen.ifc"
# Define the database file name
db_filename = 'SuD_Datenbank.db'

conn = None

try:
    # Connect to the SQLite database
    conn = sqlite3.connect(db_filename)
    c = conn.cursor()

    # Create the table if it doesn't exist
    # Added 'deletion_date' column
    c.execute('''CREATE TABLE IF NOT EXISTS ifc_objects
                 (guid TEXT, filename TEXT, added_timestamp TEXT, status TEXT DEFAULT 'active',
                  approval_architect BOOLEAN DEFAULT FALSE, approval_structure BOOLEAN DEFAULT FALSE,
                  deletion_date TEXT)''')

    # --- 1. Extract data from the updated IFC file and get creation date (ONLY from FILE_NAME header) ---
    updated_extracted_data = [] # List to hold GUIDs and filenames from the updated IFC
    updated_ifc_filename = os.path.basename(updated_file_path)
    ifc_creation_date = None # Initialize as None, will only be set if found in FILE_NAME header

    try:
        updated_ifc_file = ifcopenshell.open(updated_file_path)

        # Attempt to extract creation date ONLY from the FILE_NAME header
        ifc_header = updated_ifc_file.header
        if ifc_header and hasattr(ifc_header, 'file_name') and hasattr(ifc_header.file_name, 'time_stamp'):
            # The time_stamp is usually a string, might include timezone info or milliseconds
            timestamp_str = str(ifc_header.file_name.time_stamp) # Ensure it's a string
            try:
                # Attempt to parse various common datetime formats, including potential timezone info
                # We'll try parsing the part before any potential '+' or '.' for timezone/milliseconds
                datetime_part = timestamp_str.split('+')[0].split('.')[0]
                timestamp_obj = datetime.strptime(datetime_part, '%Y-%m-%dT%H:%M:%S')
                ifc_creation_date = timestamp_obj.strftime('%y%m%d')
                print(f"Found creation date in FILE_NAME header: {ifc_creation_date}")
            except ValueError:
                print(f"Warning: Could not parse timestamp format from FILE_NAME: {timestamp_str}. IFC creation date will not be used.")
                ifc_creation_date = None # Ensure it's None if parsing fails
        else:
             print("Warning: FILE_NAME header or time_stamp not found. IFC creation date will not be used.")
             ifc_creation_date = None


        updated_provision_for_void = updated_ifc_file.by_type("IfcVirtualElement")

        if updated_provision_for_void:
            for element in updated_provision_for_void:
                updated_extracted_data.append((element.GlobalId, updated_ifc_filename))
        else:
            print(f"No IfcVirtualElement found in the updated file: {updated_file_path}")

    except FileNotFoundError:
        print(f"Error: Updated file not found at {updated_file_path}")
        # Exit the script if the file is not found
        exit()
    except Exception as e:
        print(f"An error occurred during extraction from the updated file: {e}")
        # Exit the script on other extraction errors
        exit()

    # Create a set of GUIDs from the updated IFC data for efficient lookup
    updated_guids = set([item[0] for item in updated_extracted_data])

    # --- 2. Query existing data from the database ---
    existing_data = {} # Dictionary to hold existing GUIDs and their statuses for the filename
    try:
        c.execute("SELECT guid, status FROM ifc_objects WHERE filename = ?", (updated_ifc_filename,))
        for row in c.fetchall():
            existing_data[row[0]] = row[1]

        print(f"Found {len(existing_data)} existing objects in the database for filename '{updated_ifc_filename}'.")

    except sqlite3.Error as e:
        print(f"Database error during query: {e}")
        # Exit the script on database errors
        exit()


    # --- 3. Identify deleted objects and update their status and deletion date ---
    # Only attempt to identify and mark deleted objects if there is existing data for this filename
    if existing_data:
        deleted_guids = []
        # Use the extracted IFC creation date for deletion date if available, otherwise use current date
        deletion_timestamp = ifc_creation_date if ifc_creation_date else datetime.now().strftime('%y%m%d')

        for guid, status in existing_data.items():
            # If a GUID exists in the database but is NOT in the updated IFC data, and its status is 'active'
            if guid not in updated_guids and status == 'active':
                deleted_guids.append(guid)

        if deleted_guids:
            try:
                # Update the status to 'deleted' and set the deletion_date using the IFC creation date if available
                # If ifc_creation_date is None, the deletion_date will be set to None (or its default if defined in schema)
                c.executemany('UPDATE ifc_objects SET status = "deleted", deletion_date = ? WHERE guid = ?', [(deletion_timestamp, guid) for guid in deleted_guids])
                print(f"Successfully marked {len(deleted_guids)} objects as 'deleted' in the database.")
            except sqlite3.Error as e:
                print(f"Database error during deletion update: {e}")
                # Exit the script on database errors
                exit()
        else:
            print("No objects found in the database to mark as 'deleted' for this filename.")
    else:
        print("No existing data found for this filename, skipping deletion check.")


    # --- 4. Identify and insert new objects ---
    existing_guids_set = set(existing_data.keys()) # Use a set for efficient lookup
    new_objects_to_add = []
    # Use the extracted IFC creation date for added_timestamp if available, otherwise use current date
    added_timestamp = ifc_creation_date if ifc_creation_date else datetime.now().strftime('%y%m%d')


    for guid, filename in updated_extracted_data:
        # If a GUID from the updated IFC data is NOT in the existing database GUIDs
        if guid not in existing_guids_set:
            # Append the new object details with 'active' status and the IFC creation date if available, deletion_date is None
            new_objects_to_add.append((guid, filename, added_timestamp, 'active', False, False, None))

    if new_objects_to_add:
        try:
            c.executemany('INSERT INTO ifc_objects VALUES (?,?,?,?,?,?,?)', new_objects_to_add)
            print(f"Successfully added {len(new_objects_to_add)} new objects to the database.")
        except sqlite3.Error as e:
            print(f"Database error during new object insertion: {e}")
            # Exit the script on database errors
            exit()
    else:
        print("No new objects found in the updated IFC file to add to the database.")

    # --- 5. Commit changes ---
    conn.commit()

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

finally:
    if conn:
        conn.close()

Found creation date in FILE_NAME header: 250806
Found 26 existing objects in the database for filename 'SuD_Aussparungen.ifc'.
No objects found in the database to mark as 'deleted' for this filename.
No new objects found in the updated IFC file to add to the database.


# Freigabestatus in Datenbank updaten

In [ ]:
# Imports IfcGUIDs from an excel file to set the approval status in the database
import pandas as pd
import sqlite3

# Define the path to the Excel file containing GUIDs
# Replace with the actual path to your Excel file
csv_file_path = "SuD_Freigabe.xlsx" # Changed file extension to .xlsx

# Define the database file name
db_filename = 'SuD_Datenbank.db'

# Get input for approval type (architect or structure)
approval_type = input("Enter approval type ('architect' or 'structure'): ").lower()

# Validate approval type input
if approval_type not in ['architect', 'structure']:
    print("Invalid approval type. Please enter 'architect' or 'structure'.")
    exit()

# Determine the column to update based on approval type
approval_column = f"approval_{approval_type}"

conn = None

try:
    # Read GUIDs from the Excel file
    try:
        # Assuming the Excel has no header and GUIDs are in the first column (index 0)
        approved_guids_df = pd.read_excel(csv_file_path, header=None) # Changed to read_excel
        approved_guids = approved_guids_df[0].tolist() # Assuming GUIDs are in the first column
        print(f"Read {len(approved_guids)} GUIDs from {csv_file_path}")

    except FileNotFoundError:
        print(f"Error: Excel file not found at {csv_file_path}")
        exit()
    except Exception as e:
        print(f"An error occurred while reading the Excel file: {e}")
        exit()

    # Connect to the SQLite database
    conn = sqlite3.connect(db_filename)
    c = conn.cursor()

    updated_count = 0
    # Iterate through the approved GUIDs and update the database
    for guid in approved_guids:
        # Check if the GUID exists in the database and update the approval status
        # Using a parameterized query to prevent SQL injection
        c.execute(f"UPDATE ifc_objects SET {approval_column} = TRUE WHERE guid = ?", (guid,))
        updated_count += c.rowcount # Check if any row was updated

    # Commit the changes
    conn.commit()

    print(f"Database updated: {updated_count} objects marked as approved for {approval_type}.")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

finally:
    if conn:
        conn.close()

Enter approval type ('architect' or 'structure'): architect
Read 4 GUIDs from SuD_Freigabe.xlsx
Database updated: 4 objects marked as approved for architect.


# Freigabestatus zurück ins IFC schreiben

In [ ]:
# Writes approval status from the database file back into the ifc
import ifcopenshell
import ifcopenshell.api
import sqlite3
import pandas as pd
import os

# Define parameters for writing back to IFC
db_filename = 'SuD_Datenbank.db'
input_ifc_file = "SuD_Aussparungen.ifc"
output_ifc_file = "SuD_Aussparungen_updated.ifc" # Define the output file path

pset_name = "Planung"
architect_approval_property = "Architektur_Freigabe"
structure_approval_property = "Tragwerksplanung_Freigabe"

# Load relevant data from the database
conn = None
approved_objects_data = [] # List to store data from the database

try:
    # Connect to the SQLite database
    conn = sqlite3.connect(db_filename)
    c = conn.cursor()

    # Select GUID, approval_architect, and approval_structure for all objects
    c.execute("SELECT guid, approval_architect, approval_structure FROM ifc_objects")

    # Fetch all the results
    rows = c.fetchall()

    # Store the data in a list of dictionaries for easier lookup by GUID
    for row in rows:
        approved_objects_data.append({
            "guid": row[0],
            "approval_architect": bool(row[1]), # Convert INTEGER to BOOLEAN
            "approval_structure": bool(row[2])  # Convert INTEGER to BOOLEAN
        })

    print(f"Loaded {len(approved_objects_data)} objects from the database.")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    if conn:
        conn.close()

# You might want to convert the list of dictionaries to a dictionary keyed by GUID for faster lookup later
approved_objects_dict = {item['guid']: item for item in approved_objects_data}

# You can now use approved_objects_dict to quickly look up approval statuses by GUID
# print(approved_objects_dict) # Uncomment to see the dictionary

# Open the IFC file for modification

try:
    ifc_file = ifcopenshell.open(input_ifc_file)
    print(f"Successfully opened IFC file: {input_ifc_file}")

except FileNotFoundError:
    print(f"Error: Input IFC file not found at {input_ifc_file}")
except Exception as e:
    print(f"An error occurred while opening the IFC file: {e}")


# Match objects in the IFC file with database data by GUID

# Get all relevant elements from the IFC file (e.g., IfcVirtualElement)
ifc_objects_to_modify = ifc_file.by_type("IfcVirtualElement")

print(f"Found {len(ifc_objects_to_modify)} IfcVirtualElement(s) in the IFC file.")

matched_objects = []

for ifc_object in ifc_objects_to_modify:
    guid = ifc_object.GlobalId
    # Check if this GUID exists in the approved_objects_dict
    if guid in approved_objects_dict:
        # If a match is found, store the ifc_object and its corresponding database data
        matched_objects.append({
            "ifc_object": ifc_object,
            "db_data": approved_objects_dict[guid]
        })

print(f"Matched {len(matched_objects)} objects with data from the database.")


try:
    modified_object_count = 0 # Initialize a counter for modified objects

    # Iterate through the matched objects
    for item in matched_objects:
        ifc_element = item["ifc_object"]
        # Access 'db_data' which contains the approval information from the database
        approval_data = item["db_data"]

        # We will now write the properties for ALL matched objects, not just approved ones
        try:
            # Use ifcopenshell.api to add or get the Pset
            # We need to ensure the product is the ifc_element
            pset = ifcopenshell.api.run("pset.add_pset", ifc_file, product=ifc_element, name=pset_name)
            # print(f"Processed Pset '{pset_name}' for GUID: {ifc_element.GlobalId}") # Uncomment for detailed output

            # Prepare the properties dictionary
            properties_to_add = {}
            # Add architect approval property as a string ("True" or "False")
            properties_to_add[architect_approval_property] = str(approval_data['approval_architect'])
            # Add structure approval property as a string ("True" or "False")
            properties_to_add[structure_approval_property] = str(approval_data['approval_structure'])

            # Use ifcopenshell.api to edit the Pset and add/update properties
            ifcopenshell.api.run("pset.edit_pset", ifc_file, pset=pset, properties=properties_to_add)
            # print(f"Added/Edited properties for GUID: {ifc_element.GlobalId}") # Uncomment for detailed output
            modified_object_count += 1 # Increment the counter for each modified object

        except Exception as api_e:
            print(f"An error occurred while using ifcopenshell.api for GUID {ifc_element.GlobalId}: {api_e}")


    print(f"Successfully wrote properties to {modified_object_count} objects in the IFC file.") # Print the total count

    # --- Write the modified IFC file ---
    ifc_file.write(output_ifc_file)
    print(f"Modified IFC file saved to: {output_ifc_file}")


except Exception as e:
    print(f"An unexpected error occurred during the write-back process: {e}")
    # print(e) # Added this line to print the exception object directly

Loaded 26 objects from the database.
Successfully opened IFC file: SuD_Aussparungen.ifc
Found 22 IfcVirtualElement(s) in the IFC file.
Matched 22 objects with data from the database.
Successfully wrote properties to 22 objects in the IFC file.
Modified IFC file saved to: SuD_Aussparungen_updated.ifc


# Datenbank anschauen

In [ ]:
# Opens a sqlite database file to check its contents
import sqlite3

db_filename = 'SuD_Datenbank.db'
conn = None
try:
    conn = sqlite3.connect(db_filename)
    c = conn.cursor()

    # Select all rows from the ifc_objects table
    c.execute("SELECT * FROM ifc_objects")

    # Get the column names from the cursor description
    column_headers = [description[0] for description in c.description]

    # Print the column headers
    print(column_headers)

    # Fetch all the results
    rows = c.fetchall()

    # Print the rows
    for row in rows:
        print(row)

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    if conn:
        conn.close()

['guid', 'filename', 'added_timestamp', 'status', 'approval_architect', 'approval_structure', 'deletion_date']
('1MSlHG7wz54ww1tgs9pFDk', 'SuD_Aussparungen.ifc', '250806', 'active', 1, 0, None)
('1MSlHG7wz54ww1tgs9pFDj', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3XJ', 'SuD_Aussparungen.ifc', '250806', 'deleted', 0, 0, '250806')
('1MSlHG7wz54ww1tgs9p3Zz', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3ia', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3jP', 'SuD_Aussparungen.ifc', '250806', 'active', 1, 0, None)
('1MSlHG7wz54ww1tgs9p3k3', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3fu', 'SuD_Aussparungen.ifc', '250806', 'active', 1, 0, None)
('1MSlHG7wz54ww1tgs9p3q$', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3q_', 'SuD_Aussparungen.ifc', '250806', 'active', 0, 0, None)
('1MSlHG7wz54ww1tgs9p3qx', 'SuD_Aussparungen.ifc', '25

# Datenbank als Excel exportieren

In [ ]:
# Exports the sqlite database file to an excel file for userfriendly review of the data
import sqlite3
import pandas as pd

db_filename = 'SuD_Datenbank.db'
conn = None

try:
    conn = sqlite3.connect(db_filename)

    # Read data from the table into a pandas DataFrame
    df = pd.read_sql_query("SELECT * FROM ifc_objects", conn)

    # Close the database connection
    conn.close()

    # Save the DataFrame to an Excel file (requires openpyxl library)
    df.to_excel('SuD_Datenbank.xlsx', index=False)
    print("Data successfully exported to SuD_Datenbank.xlsx")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    if conn:
        conn.close()

Data successfully exported to SuD_Datenbank.xlsx
